In [15]:
import pandas as pd
import numpy as np

# Load data
df = pd.read_csv("Traffic_Volume_Counts_20251111.csv")

# Fix date format (NYC is usually month/day/year)
df["Date"] = pd.to_datetime(df["Date"], format="%m/%d/%Y", errors="coerce")

# Identify hourly columns
hourly_cols = [col for col in df.columns if ":" in col]
print("Hour columns found:", len(hourly_cols))

# Melt hourly cols
df_long = df.melt(
    id_vars=["ID", "SegmentID", "Roadway Name", "From", "To", "Direction", "Date"],
    value_vars=hourly_cols,
    var_name="Hour",
    value_name="Traffic_Count"
)

# Fixing issues with strings in traffic count by eliminating them
df_long["Traffic_Count"] = (
    df_long["Traffic_Count"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .astype(float)
)

# hour parser
def extract_start_time(hour_string):
    try:
        start = hour_string.split("-")[0].strip()      # "12:00"
        end = hour_string.split("-")[1].strip()        # "1:00 AM"

        # If start doesn't include AM/PM, borrow from the ending
        if "AM" not in start and "PM" not in start:
            suffix = end[-2:]                         # "AM" or "PM"
            start = f"{start} {suffix}"               # "12:00 AM"

        # Convert to datetime time
        return pd.to_datetime(start, format="%I:%M %p")
    except:
        return pd.NaT


df_long["Start_Time"] = df_long["Hour"].apply(extract_start_time)

# Drop invalid rows
df_long = df_long.dropna(subset=["Date", "Start_Time"])

# Extract hours and minutes from Start_Time
df_long["Start_Hour"] = df_long["Start_Time"].dt.hour
df_long["Start_Minute"] = df_long["Start_Time"].dt.minute

# Build Datetime safely
df_long["Datetime"] = (
    df_long["Date"].dt.normalize()
    + pd.to_timedelta(df_long["Start_Hour"], unit="h")
    + pd.to_timedelta(df_long["Start_Minute"], unit="m")
)

df_long = df_long.sort_values(by=["SegmentID", "Datetime"]).reset_index(drop=True)



Hour columns found: 24


In [17]:
# Baseline model, linear regression

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

# Starting with preparing to split
X = df_long[["Start_Hour"]]
y = df_long["Traffic_Count"]

# Still having weird issue with NAs? This is a precaution, may not be needed
mask = X.notna().all(axis=1) & y.notna()
X = X[mask]
y = y[mask]
df_clean = df_long[mask]

# Splitting by SegmentID (street) to keep streets either in test or train, not both
segments = df_long["SegmentID"].unique()

train_segments, test_segments = train_test_split(segments, test_size=0.2, random_state=42)

print("Train segments:", len(train_segments))
print("Test segments:", len(test_segments))


# Extra precaution to avoid error
train_mask = df_clean["SegmentID"].isin(train_segments)
test_mask  = df_clean["SegmentID"].isin(test_segments)

X_train = X[train_mask]
y_train = y[train_mask]
X_test  = X[test_mask]
y_test  = y[test_mask]

# Now, data is split, so moving to the actual baseline model
lr = LinearRegression()
lr.fit(X_train,y_train)

lr_pred = lr.predict(X_test)
baseline_mse = mean_squared_error(y_test, lr_pred)


print("Baseline Model MSE:", baseline_mse)


Train segments: 1564
Test segments: 392
Baseline Model MSE: 440694.62067069154


In [ ]:
# start training an RNN

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import RNN, Dense, SimpleRNN, LSTM

model = Sequential()
model.add(LSTM(50, activation='relu', input_shape=(, )))
model.add(Dense(1))

model.compile(
    loss='categorical_crossentropy',
    optimizer=RMSprop(learning_rate=0.01),
    metrics=['categorical_crossentropy', 'accuracy']
)


